# Regional UKCP18 CPM NetCDF export

Select a region and period, then export Gregorian rainfall amounts in mm with spatial coordinates, CRS information and time bounds.

This is an optional preprocessing step. The DSS converter can also read CPM source files and clip them directly.

In [ ]:
from pathlib import Path
import json
from dss_pipeline import main


## Configuration

Set the input, output and boundary paths. Date-only end values include the whole day; datetime end values are exclusive.

For native rotated grids, add `target_crs='EPSG:27700'` and `target_cell_size=2200`. Set `inspect=True` to display source metadata without writing output.

In [ ]:
CONFIG = {'input_dir': 'D:/climate/ukcp18-cpm',
 'input_pattern': 'pr*.nc',
 'output_dir': 'D:/hec-ras/regional-netcdf',
 'variable': 'pr',
 'boundary': 'D:/boundaries/forth.shp',
 'name': 'forth',
 'start': '2020-02-03',
 'end': '2020-02-07'}

RUN = False


In [ ]:
def cli_arguments(config):
    arguments = []
    for key, value in config.items():
        flag = "--" + key.replace("_", "-")
        if key == "window":
            for row in value:
                arguments.extend([flag, *map(str, row)])
        elif key == "select":
            for item in value:
                arguments.extend([flag, str(item)])
        elif isinstance(value, bool):
            if value:
                arguments.append(flag)
        elif isinstance(value, list):
            arguments.extend([flag, *map(str, value)])
        elif value is not None:
            arguments.extend([flag, str(value)])
    return arguments


## Run

Set `RUN=True` after editing the configuration. Source files are read-only. Use a new output directory or set `overwrite=True` to replace an existing output.

In [ ]:
if RUN:
    jobs = main('cpm', cli_arguments(CONFIG), clip=True)
    for metadata_path in jobs:
        metadata = json.loads(Path(metadata_path).read_text(encoding='ascii'))
        print({
            'start': metadata['start'],
            'end_exclusive': metadata['end_exclusive'],
            'intervals': metadata['record_count'],
            'cell_size_m': metadata['cell_size'],
        })
else:
    print('Edit CONFIG and set RUN=True to start.')
